In [ ]:
"""
Dataset subset extraction for the guru-RL-92k math problems.

Download the math parquet file from:
https://huggingface.co/datasets/LLM360/guru-RL-92k/blob/main/train/math__combined_54.4k.parquet

Save it to: data/math__combined_54.4k.parquet
"""

import numpy as np
import pandas as pd

df_guru_math = pd.read_parquet("data/math__combined_54.4k.parquet")
df_guru_math['qwen2.5_7b_pass_rate'].hist()
df_guru_math['qwen3_30b_pass_rate'].hist()
df_guru_math.count()

In [ ]:
print(f'original size of math subset: {df_guru_math.shape[0]}')
df = df_guru_math[df_guru_math['ability'] == 'math'].copy()  # explicit copy to break view chain
df = df[df['is_unique'] == True]
# df = df[df['reward_model']['style'] == 'rule']
df = df[df['qwen3_30b_pass_rate'] != df['qwen2.5_7b_pass_rate']]
df = df[df['qwen3_30b_pass_rate'] > 0.5]   # at least some improvement is viable
df = df[df['qwen2.5_7b_pass_rate'] < 0.7]  # make sure no easy ones (this has no effect here since max is 0.54)
print(f'size of trainable math subset: {df.shape[0]}')

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4, 3))
bins = 10  # or a list of bin edges

w2 = np.ones(len(df)) * 100.0 / len(df)
df['qwen2.5_7b_pass_rate'].hist(ax=ax, bins=bins, alpha=1, label='qwen2.5 7b', weights=w2)

w3 = np.ones(len(df)) * 100.0 / len(df)
df['qwen3_30b_pass_rate'].hist(ax=ax, bins=bins, alpha=1, label='qwen3 30b', weights=w3)

ax.set_xlabel('Pass Rate')
ax.set_ylabel('Percent of rows (%)')
ax.set_title('Pass Rate Distribution for Trainable Math Subset')
ax.legend()
fig.savefig("data/trainable_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import pandas as pd

N=1000
n_bins=10
random_state=42
col = 'qwen2.5_7b_pass_rate'

# Save the full trainable data before sampling (for plotting later)
df_full_trainable = df.copy()

# Quantile bins; duplicates="drop" handles repeated values like many zeros
df["_bin"] = pd.qcut(df[col], q=n_bins, duplicates="drop")

# Proportions per bin in the full df
bin_counts = df["_bin"].value_counts(sort=False)
props = bin_counts / len(df)

# Exact integer quotas that sum to N (largest remainder method)
raw = props.values * N
base = np.floor(raw).astype(int)
remainder = N - base.sum()
if remainder > 0:
    order = np.argsort(-(raw - base))
    base[order[:remainder]] += 1

quotas = pd.Series(base, index=props.index)

rng = np.random.default_rng(random_state)
parts = []
for b, k in quotas.items():
    pool = df[df["_bin"] == b]
    if k > len(pool):
        # fallback: sample with replacement only if absolutely necessary
        parts.append(pool.sample(n=k, replace=True, random_state=int(rng.integers(0, 2**32-1))))
    elif k > 0:
        parts.append(pool.sample(n=k, replace=False, random_state=int(rng.integers(0, 2**32-1))))

sampled = pd.concat(parts).sample(frac=1, random_state=random_state).drop(columns=["_bin"])
sampled['data_source'] = 'math'  # to assure VeRL pipeline compatibility for the reward function

# Bucket-count sanity check (using sampled itself; no df2.loc needed)
s = sampled["qwen2.5_7b_pass_rate"]
counts = {
    "zero": int(s.eq(0).sum()),
    "(0,0.1]": int((s.gt(0) & s.le(0.1)).sum()),
    "(0.1,0.2]": int((s.gt(0.1) & s.le(0.2)).sum()),
    "(0.2,0.3]": int((s.gt(0.2) & s.le(0.3)).sum()),
    "(0.3,0.4]": int((s.gt(0.3) & s.le(0.4)).sum()),
    "(0.4,1]": int((s.gt(0.4)).sum()),
}
counts, sum(counts.values())

In [ ]:
# Verify the columns available in the sampled dataframe
print("Columns in sampled dataframe:")
print(sampled.columns.tolist())
print(f"\nSample row:")
print(sampled.iloc[0].to_dict())


In [ ]:
# Save training data in verl-compatible format
# Required columns: data_source, prompt, reward_model
# Extra columns (pass rates, etc.) are kept for analysis - verl ignores them

import os
os.makedirs("data", exist_ok=True)

# Verify required columns exist
required_cols = ['data_source', 'prompt', 'reward_model']
missing = [c for c in required_cols if c not in sampled.columns]
if missing:
    print(f"❌ Missing required columns: {missing}")
    print("Available columns:", sampled.columns.tolist())
else:
    # Save the full dataframe (verl uses what it needs, extra cols for analysis)
    train_path = "data/train.parquet"
    sampled.to_parquet(train_path, index=False)
    print(f"✅ Saved training data to {train_path}")
    print(f"   - {len(sampled)} problems")
    print(f"   - Columns: {sampled.columns.tolist()}")


In [ ]:
# Plot pass rate distributions: Combined view with full/sampled counts
# Single plot showing both models with dual count labels

COLORS = {'qwen2.5': "#00B3FF", 'qwen3': "#FF553B"}  # Blue and Red

def get_bucket_counts(data, col):
    """Compute bucket counts for a given column."""
    s = data[col]
    return {
        "[0,0.2]": int(s.le(0.2).sum()),
        "(0.2,0.4]": int((s.gt(0.2) & s.le(0.4)).sum()),
        "(0.4,0.6]": int((s.gt(0.4) & s.le(0.6)).sum()),
        "(0.6,0.8]": int((s.gt(0.6) & s.le(0.8)).sum()),
        "(0.8,1]": int((s.gt(0.8) & s.le(1.0)).sum()),
    }

# Get counts for both datasets
full_7b = get_bucket_counts(df_full_trainable, 'qwen2.5_7b_pass_rate')
full_30b = get_bucket_counts(df_full_trainable, 'qwen3_30b_pass_rate')
samp_7b = get_bucket_counts(sampled, 'qwen2.5_7b_pass_rate')
samp_30b = get_bucket_counts(sampled, 'qwen3_30b_pass_rate')

bucket_names = list(full_7b.keys())
x = np.arange(len(bucket_names))
width = 0.45

fig, ax = plt.subplots(figsize=(5, 3))

# Use sampled counts for bar heights (cleaner scale)
vals_7b = list(samp_7b.values())
vals_30b = list(samp_30b.values())

bars1 = ax.bar(x - width/2, vals_7b, width, label='Qwen2.5-7B', color=COLORS['qwen2.5'], alpha=0.85)
bars2 = ax.bar(x + width/2, vals_30b, width, label='Qwen3-30B', color=COLORS['qwen3'], alpha=0.85)

# Add dual count labels: "full / sampled" on each bar
for i, bar in enumerate(bars1):
    height = bar.get_height()
    full_val = list(full_7b.values())[i]
    samp_val = list(samp_7b.values())[i]
    label = f"{full_val}/{samp_val}"
    ax.text(bar.get_x() + bar.get_width()/2, height + 2, label,
            ha='center', va='bottom', fontsize=7, fontweight='bold', color=COLORS['qwen2.5'])

for i, bar in enumerate(bars2):
    height = bar.get_height()
    full_val = list(full_30b.values())[i]
    samp_val = list(samp_30b.values())[i]
    label = f"{full_val}/{samp_val}"
    ax.text(bar.get_x() + bar.get_width()/2, height + 2, label,
            ha='center', va='bottom', fontsize=7, fontweight='bold', color=COLORS['qwen3'])

ax.set_xlabel('Pass Rate Bucket', fontsize=8)
ax.set_ylabel('')  # No y-axis label
ax.set_title(f'Pass Rate Distribution\nFull(trainable): {len(df_full_trainable):,} → Sampled: {len(sampled):,} problems', 
             fontweight='bold', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(bucket_names, fontsize=8)
ax.set_yticks([])  # Remove y-axis ticks
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.legend(loc='upper center', fontsize=8)
ax.grid(axis='y', alpha=0.3)

# Add headroom for labels
ax.set_ylim(top=ax.get_ylim()[1] * 1.25)

# Add note explaining label format
ax.text(0.02, 0.87, 'Labels: full(trainable) / sampled', transform=ax.transAxes, 
        fontsize=7.5, va='top', ha='left', style='italic', color='gray')

plt.tight_layout()
plt.savefig("data/train_distribution.png", dpi=150, bbox_inches='tight')
print("✅ Saved distribution plot to data/train_distribution.png")
plt.show()


## Evaluation Data Preparation

**Evaluation Strategy:**

- **Online eval** (during training): Full MATH-500 + Full AIME24 (530 problems total)
  - Purpose: Complete evaluation during training - no need for separate offline eval
  - Combined into single file: `eval_online.parquet`
  - Configurable fractions via `online_math500_fraction` and `online_aime24_fraction` (default: 1.0 = full)
  
- **Offline eval** (archival): Separate files for each benchmark
  - `eval_offline_math500.parquet` - Full MATH-500 (500 problems)
  - `eval_offline_aime24.parquet` - Full AIME24 (30 problems)
  - These are saved for archival purposes but not needed if using full online eval

**Data Sources:**
- MATH-500: https://huggingface.co/datasets/HuggingFaceH4/MATH-500
- AIME24: https://huggingface.co/datasets/AI-MO/aimo-validation-aime

In [ ]:
# Prepare evaluation datasets from HuggingFace

import pandas as pd
from datasets import load_dataset

def convert_to_verl_format(example, data_source):
    """Convert HuggingFace dataset example to verl format."""
    # Handle different dataset formats for prompt
    if 'problem' in example:
        prompt_text = example['problem']
    elif 'question' in example:
        prompt_text = example['question']
    else:
        prompt_text = str(example.get('prompt', ''))
    
    # Get ground truth - use 'answer' (the final answer) not 'solution' (step-by-step)
    # MATH-500 has both fields; we need the answer for reward comparison
    if 'answer' in example:
        gt = str(example['answer'])
    elif 'solution' in example:
        gt = example['solution']
    else:
        gt = ''
    
    return {
        'data_source': data_source,
        'prompt': [{'role': 'user', 'content': prompt_text}],
        'reward_model': {'style': 'rule', 'ground_truth': gt}
    }

# Load MATH-500
print("Loading MATH-500 from HuggingFace...")
math500_df = None
try:
    math500 = load_dataset("HuggingFaceH4/MATH-500", split="test")
    math500_records = [convert_to_verl_format(ex, "math500") for ex in math500]
    math500_df = pd.DataFrame(math500_records)
    print(f"  ✅ Loaded {len(math500_df)} problems from MATH-500")
except Exception as e:
    print(f"  ❌ Could not load MATH-500: {e}")

# Load AIME24
print("\nLoading AIME24 from HuggingFace...")
aime24_df = None
try:
    # AIME24 from AI-MO validation set
    aime24 = load_dataset("AI-MO/aimo-validation-aime", split="train")
    aime24_records = [convert_to_verl_format(ex, "aime24") for ex in aime24]
    aime24_df = pd.DataFrame(aime24_records)
    print(f"  ✅ Loaded {len(aime24_df)} problems from AIME24")
except Exception as e:
    print(f"  ❌ Could not load AIME24: {e}")
    print("  Trying alternative: lighteval/MATH AIME subset...")
    try:
        # Fallback: try to get AIME problems from another source
        aime = load_dataset("lighteval/MATH", "all", split="test")
        aime_filtered = [ex for ex in aime if 'aime' in ex.get('type', '').lower() or 'AIME' in ex.get('problem', '')]
        if aime_filtered:
            aime24_records = [convert_to_verl_format(ex, "aime24") for ex in aime_filtered[:30]]
            aime24_df = pd.DataFrame(aime24_records)
            print(f"  ✅ Loaded {len(aime24_df)} AIME problems from lighteval/MATH")
    except Exception as e2:
        print(f"  ❌ Fallback also failed: {e2}")

print(f"\n📊 Summary:")
print(f"   MATH-500: {len(math500_df) if math500_df is not None else 0} problems")
print(f"   AIME24:   {len(aime24_df) if aime24_df is not None else 0} problems")


In [ ]:
# Save evaluation datasets
# 
# Parameters for controlling eval data fractions:
#   - online_math500_fraction: Fraction of MATH-500 to include in online eval (default: 1.0 = full)
#   - online_aime24_fraction: Fraction of AIME24 to include in online eval (default: 1.0 = full)
#
# By default, online eval uses FULL data for both benchmarks.
# Offline eval files are still generated separately for archival purposes.

import os
os.makedirs("data", exist_ok=True)

# ============================================================
# PARAMETERS: Control eval data fractions
# ============================================================
online_math500_fraction = 1.0   # 1.0 = full MATH-500 (500 problems)
online_aime24_fraction = 1.0    # 1.0 = full AIME24 (30 problems)

# ============================================================
# OFFLINE EVAL: Separate files per benchmark (for archival)
# ============================================================

# Save full MATH-500 for offline eval
if math500_df is not None:
    offline_math_path = "data/eval_offline_math500.parquet"
    math500_df.to_parquet(offline_math_path, index=False)
    print(f"✅ Saved offline MATH-500 to {offline_math_path}")
    print(f"   - {len(math500_df)} problems")

# Save full AIME24 for offline eval  
if aime24_df is not None:
    offline_aime_path = "data/eval_offline_aime24.parquet"
    aime24_df.to_parquet(offline_aime_path, index=False)
    print(f"✅ Saved offline AIME24 to {offline_aime_path}")
    print(f"   - {len(aime24_df)} problems")

# ============================================================
# ONLINE EVAL: Combined file (configurable fractions)
# ============================================================

online_parts = []

# Add AIME24 (configurable fraction, default = full)
if aime24_df is not None:
    if online_aime24_fraction >= 1.0:
        aime24_online = aime24_df
    else:
        n_aime = max(1, int(len(aime24_df) * online_aime24_fraction))
        aime24_online = aime24_df.sample(n=n_aime, random_state=42)
    online_parts.append(aime24_online)
    print(f"\n📊 Online eval: {len(aime24_online)} AIME24 problems ({online_aime24_fraction*100:.0f}%)")

# Add MATH-500 (configurable fraction, default = full)
if math500_df is not None:
    if online_math500_fraction >= 1.0:
        math500_online = math500_df
    else:
        n_math = max(1, int(len(math500_df) * online_math500_fraction))
        math500_online = math500_df.sample(n=n_math, random_state=42)
    online_parts.append(math500_online)
    print(f"📊 Online eval: {len(math500_online)} MATH-500 problems ({online_math500_fraction*100:.0f}%)")

# Combine and save
if online_parts:
    online_df = pd.concat(online_parts, ignore_index=True)
    online_path = "data/eval_online.parquet"
    online_df.to_parquet(online_path, index=False)
    print(f"\n✅ Saved online eval to {online_path}")
    print(f"   - {len(online_df)} total problems")
    print(f"   - data_source breakdown: {online_df['data_source'].value_counts().to_dict()}")

# ============================================================
# SUMMARY
# ============================================================
print("\n" + "="*60)
print("📋 EVALUATION DATA SUMMARY")
print("="*60)
print("\nOnline eval (during training):")
print(f"   data/eval_online.parquet - {len(online_df) if online_parts else 0} problems")
print(f"   MATH-500: {online_math500_fraction*100:.0f}% | AIME24: {online_aime24_fraction*100:.0f}%")
print("\nOffline eval (archival - separate benchmarks):")
print("   data/eval_offline_math500.parquet  - Full MATH-500")
print("   data/eval_offline_aime24.parquet   - Full AIME24")
print("="*60)


## Statistics for Paper Discussion

Key statistics about the training dataset that can be used in the paper.


In [ ]:
# Statistics for paper discussion
print("=" * 60)
print("TRAINING SET STATISTICS")
print("=" * 60)

print(f"\n📊 Dataset Size:")
print(f"   Original guru-RL-92k math subset: {len(df_guru_math):,} problems")
print(f"   After filtering (trainable):      {len(df):,} problems")
print(f"   Final training set:               {len(sampled):,} problems")

print(f"\n📈 Pass Rate Statistics (Qwen2.5-7B):")
pr = sampled['qwen2.5_7b_pass_rate']
print(f"   Min:    {pr.min():.3f}")
print(f"   Max:    {pr.max():.3f}")
print(f"   Mean:   {pr.mean():.3f}")
print(f"   Median: {pr.median():.3f}")
print(f"   Std:    {pr.std():.3f}")

print(f"\n🎯 Difficulty Distribution:")
for bucket, count in counts.items():
    print(f"   {bucket:12s}: {count:3d} problems ({100*count/len(sampled):.1f}%)")

print(f"\n📐 Filtering Criteria:")
print(f"   - ability == 'math'")
print(f"   - is_unique == True")
print(f"   - qwen3_30b_pass_rate != qwen2.5_7b_pass_rate (room for improvement)")
print(f"   - qwen3_30b_pass_rate > 0.5 (at least some improvement viable)")
print(f"   - qwen2.5_7b_pass_rate < 0.7 (not too easy)")

print(f"\n🔄 Sampling Strategy:")
print(f"   - Stratified sampling by pass rate quantiles")
print(f"   - {n_bins} bins, largest remainder method for integer quotas")
print(f"   - Random seed: {random_state}")
print("=" * 60)
